In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.metrics import r2_score

In [ ]:
#  Configuration 
DATA_PATH = "kc_house_data.csv"   # change if your file is named differently
OUT_DIR = "project_outputs"
os.makedirs(OUT_DIR, exist_ok=True)

def save_text(filename, text):
    with open(os.path.join(OUT_DIR, filename), "w", encoding="utf-8") as f:
        f.write(text)

In [ ]:
def main():
    #  Load dataset
    print("Loading dataset from", DATA_PATH)
    df = pd.read_csv(DATA_PATH)
    print("Dataset loaded. Shape:", df.shape)
    
    # Show first rows (optional)
    df_head_path = os.path.join(OUT_DIR, "head.csv")
    df.head().to_csv(df_head_path, index=False)
    print("Saved head to", df_head_path)
    

In [ ]:
 # Task A: Display dtypes and save screenshot-ready artifact
   
    dtypes = df.dtypes
    print("\nData types of each column:\n", dtypes)
    save_text("dtypes.txt", dtypes.to_string())
    
    # Also create a small PNG that shows code + dtypes (for screenshot submission,
    # you can open dtypes.txt and the script side-by-side). We'll create a table plot.
    fig, ax = plt.subplots(figsize=(8, max(2, len(dtypes)*0.25)))
    ax.axis('off')
    table = ax.table(cellText=[[str(v)] for v in dtypes.values],
                     rowLabels=dtypes.index.tolist(),
                     colLabels=["dtype"],
                     loc='center')
    table.auto_set_font_size(False)
    table.set_fontsize(8)
    plt.title("Column dtypes")
    plt.tight_layout()
    dtypes_png = os.path.join(OUT_DIR, "dtypes_table.png")
    plt.savefig(dtypes_png, dpi=150)
    plt.close()
    print("Saved dtypes table image to", dtypes_png)
    

In [ ]:
# Task B: Drop 'id' and 'Unnamed: 0' then describe()
   
    drop_cols = [c for c in ["id", "Unnamed: 0"] if c in df.columns]
    if drop_cols:
        df.drop(columns=drop_cols, axis=1, inplace=True)
        print("Dropped columns:", drop_cols)
    else:
        print("No 'id' or 'Unnamed: 0' columns found to drop.")
    desc = df.describe(include='all')
    desc_path = os.path.join(OUT_DIR, "describe.csv")
    desc.to_csv(desc_path)
    print("Saved describe() output to", desc_path)
    save_text("describe.txt", desc.to_string())
    

In [ ]:
# Task C: value_counts of unique floors -> to_frame()
    
    if "floors" in df.columns:
        floors_vc = df["floors"].value_counts().sort_index()
        floors_df = floors_vc.to_frame(name="count").reset_index().rename(columns={"index":"floors"})
        floors_df_path = os.path.join(OUT_DIR, "floors_value_counts.csv")
        floors_df.to_csv(floors_df_path, index=False)
        print("\nSaved floors value_counts to", floors_df_path)
        save_text("floors_value_counts.txt", floors_df.to_string(index=False))
    else:
        print("Column 'floors' not present in dataset.")
    

In [ ]:
# Task D: Boxplot to compare price outliers by waterfront
    
    if "waterfront" in df.columns and "price" in df.columns:
        plt.figure(figsize=(6,6))
        sns.boxplot(x="waterfront", y="price", data=df)
        plt.title("Price distribution by Waterfront (0 = no, 1 = yes)")
        plt.xlabel("waterfront")
        plt.ylabel("price")
        boxplot_path = os.path.join(OUT_DIR, "boxplot_price_by_waterfront.png")
        plt.tight_layout()
        plt.savefig(boxplot_path, dpi=150)
        plt.close()
        print("Saved boxplot to", boxplot_path)
    else:
        print("Cannot create boxplot - 'waterfront' or 'price' missing.")
    

In [ ]:
# Task E: regplot sqft_above vs price
    
    if "sqft_above" in df.columns and "price" in df.columns:
        plt.figure(figsize=(7,5))
        sns.regplot(x="sqft_above", y="price", data=df, scatter_kws={'s':10}, line_kws={'linewidth':1})
        plt.title("Price vs sqft_above (regplot)")
        plt.xlabel("sqft_above")
        plt.ylabel("price")
        regplot_path = os.path.join(OUT_DIR, "regplot_price_sqft_above.png")
        plt.tight_layout()
        plt.savefig(regplot_path, dpi=150)
        plt.close()
        print("Saved regplot to", regplot_path)
    else:
        print("Cannot create regplot - 'sqft_above' or 'price' missing.")
    
    # --------------------------
    # Prepare data for modeling: drop rows with NA in selected features/target
    # --------------------------
    target = "price"
    features_list = ["floors","waterfront","lat","bedrooms","sqft_basement","view",
                     "bathrooms","sqft_living15","sqft_above","grade","sqft_living"]
    available_features = [f for f in features_list if f in df.columns]
    print("\nAvailable features for modeling:", available_features)
    modeling_df = df.dropna(subset=[target] + available_features)
    
    # Train/test split (for reproducibility)
    X = modeling_df[available_features].copy()
    y = modeling_df[target].copy()
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    print("Train/test split sizes:", X_train.shape, X_test.shape)
    
    results = {}

In [ ]:

    # Task F: Linear regression with single feature 'sqft_living'
    
    if "sqft_living" in X_train.columns:
        lr = LinearRegression()
        X_train_sq = X_train[["sqft_living"]].values
        X_test_sq = X_test[["sqft_living"]].values
        lr.fit(X_train_sq, y_train)
        y_pred = lr.predict(X_test_sq)
        r2_single = r2_score(y_test, y_pred)
        results['linear_sqft_living_r2'] = r2_single
        print("\nR^2 for LinearRegression(sqft_living):", r2_single)
        save_text("r2_linear_sqft_living.txt", f"R2: {r2_single}\\n")
    else:
        print("Feature 'sqft_living' not available for single-feature regression.")
    

In [ ]:

    # Task G: Linear regression with multiple features
    
    if len(available_features) > 0:
        lr_multi = LinearRegression()
        lr_multi.fit(X_train[available_features], y_train)
        y_pred_multi = lr_multi.predict(X_test[available_features])
        r2_multi = r2_score(y_test, y_pred_multi)
        results['linear_multi_r2'] = r2_multi
        print("R^2 for LinearRegression(multiple features):", r2_multi)
        save_text("r2_linear_multi.txt", f"R2: {r2_multi}\\n")
    else:
        print("No available features for multivariate regression.")
    

In [ ]:
# Task H: Pipeline: scaling -> polynomial transform -> linear regression
    
    # We'll use degree=2 for polynomial features (you can change this)
    poly_degree = 2
    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("poly", PolynomialFeatures(degree=poly_degree, include_bias=False)),
        ("lin", LinearRegression())
    ])
    pipe.fit(X_train[available_features], y_train)
    y_pred_pipe = pipe.predict(X_test[available_features])
    r2_pipe = r2_score(y_test, y_pred_pipe)
    results['pipeline_poly_r2'] = r2_pipe
    print(f"R^2 for Pipeline(StandardScaler->Poly(degree={poly_degree})->LinearRegression):", r2_pipe)
    save_text("r2_pipeline_poly.txt", f"R2: {r2_pipe}\\n")
    

In [ ]:
# Task I: Ridge regression alpha=0.1 on original features
   
    ridge = Ridge(alpha=0.1)
    ridge.fit(X_train[available_features], y_train)
    y_pred_ridge = ridge.predict(X_test[available_features])
    r2_ridge = r2_score(y_test, y_pred_ridge)
    results['ridge_r2'] = r2_ridge
    print("R^2 for Ridge(alpha=0.1) on original features:", r2_ridge)
    save_text("r2_ridge.txt", f"R2: {r2_ridge}\\n")
    

In [ ]:
# Task J: Second-order polynomial transform followed by Ridge(alpha=0.1)
    
    poly = PolynomialFeatures(degree=2, include_bias=False)
    X_train_poly = poly.fit_transform(X_train[available_features])
    X_test_poly = poly.transform(X_test[available_features])
    ridge2 = Ridge(alpha=0.1)
    ridge2.fit(X_train_poly, y_train)
    y_pred_ridge2 = ridge2.predict(X_test_poly)
    r2_ridge2 = r2_score(y_test, y_pred_ridge2)
    results['ridge_poly_r2'] = r2_ridge2
    print("R^2 for Ridge(alpha=0.1) after 2nd-order polynomial transform:", r2_ridge2)
    save_text("r2_ridge_poly.txt", f"R2: {r2_ridge2}\\n")